# 02 — Intent classification benchmark

**Phase B** (`docs/roadmap.md`). The comparable baseline the drift paper rests
on: intent classification on Ninapro DB6, scored under **four evaluation
protocols** rather than one. Feeds `reborn.ml.intent.IntentClassifier` once a
model is worth wiring in.

| Protocol | Train | Test | What it measures |
|---|---|---|---|
| Within-session | part of session *k* | rest of session *k* | Ceiling |
| **Cross-session** | earlier sessions | a later session | **Drift** |
| Cross-subject | other subjects | held-out subject | Worst case |
| Random-shuffle | shuffled windows | shuffled windows | **Control, not a result** |

**On the random-shuffle control.** Windows overlap by 150 ms at a 200 ms/50 ms
configuration, so shuffling puts near-copies of the same window on both sides of
the split. How much that inflates the score depends on **model capacity**: a
high-capacity model (kNN, a deep net) can exploit the near-duplicates, while LDA
on ten features has nothing to memorise with and may show little or no gap. So
this row is a control on the *protocol-plus-model pair*, not a fixed correction
factor — and a small gap under LDA is not licence to drop it when a larger model
is adopted later. Report whatever it shows, including "no inflation here".

> **Scope.** This notebook establishes *accuracy* across protocols. What happens
> to **confidence calibration** under the same drift — the part that connects to
> the architecture paper — is `03_drift_fewshot`. The calibration and
> unsafe-assist columns appear here already because the harness computes them for
> free; read them as a preview, not as this notebook's claim.

## Setup

Needs scikit-learn (`pip install -e ".[ml]"`) on top of the phase-B environment.
Run with `py -3.11` (`docs/research/phase-b-plan.md` §10).

In [ ]:
import csv
import json
import time
from pathlib import Path

import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression

from reborn.data.evaluation import evaluate_splits, summarize
from reborn.data.features import FEATURE_NAMES, feature_matrix, standardize
from reborn.data.loaders import NinaproDB6Loader
from reborn.data.pipeline import PreprocessConfig, build_window_set
from reborn.data.splits import (
    cross_session_splits,
    cross_subject_splits,
    random_window_split,
    within_session_splits,
)

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = REPO / "experiments" / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

CONFIG = json.loads((REPO / "experiments" / "configs" / "ninapro_db6_qc.json").read_text())
QC_KWARGS = CONFIG["qc_kwargs"]
MONTAGE = tuple(CONFIG["channels"]["montage"])
PRE = CONFIG["preprocess"]

config = PreprocessConfig(
    target_sample_rate=PRE["target_sample_rate_hz"],
    bandpass_hz=tuple(PRE["bandpass_hz"]),
    notch_hz=PRE["notch_hz"],
    window_ms=PRE["window_ms"],
    stride_ms=PRE["stride_ms"],
    pure_windows=PRE["pure_windows"],
    qc_kwargs=QC_KWARGS,
)
print("config fingerprint:", config.fingerprint())
print("features:", FEATURE_NAMES)

## 1. Build the window set

One pass: load → resample → filter → window → QC gate. Windows that fail QC never
reach the classifier, which is the runtime order — the safety layer decides
whether EMG is trustworthy *before* anything acts on it.

**Cost.** Two subjects x 10 sessions is ~260k windows and takes minutes. The
cache below means you pay that once; delete the `.npz` to force a rebuild.
Cross-subject needs at least two subjects, so `s01` alone will skip that protocol.

In [ ]:
from reborn.data.pipeline import load_window_set, save_window_set

SUBJECTS = ["s01", "s02"]
SESSIONS = None            # None = every session

cache = REPO / "data" / "cache" / f"db6_{'_'.join(SUBJECTS)}_{config.fingerprint()}.npz"
loader = NinaproDB6Loader(REPO / "data" / "ninapro_db6", channels=MONTAGE)

if cache.exists():
    window_set = load_window_set(cache)
    print(f"loaded cache: {cache.name}")
else:
    started = time.time()
    window_set = build_window_set(loader.load(subjects=SUBJECTS, sessions=SESSIONS), config)
    save_window_set(window_set, cache, config)
    print(f"built and cached in {time.time() - started:.0f} s")

sessions = sorted(map(str, set(window_set.session_ids)))
print(f"\n{window_set.n_windows} windows, {window_set.n_channels} channels")
print(f"subjects: {sorted(map(str, set(window_set.subject_ids)))}")
print(f"sessions: {len(sessions)}  {sessions}")
print(f"QC rejected {window_set.qc.rejection_rate:.2%} before this point")

# Drift needs elapsed time to accumulate. Loading only adjacent sessions produces
# a small cross-session gap for reasons that have nothing to do with the dataset.
if len(sessions) < 4:
    print(
        f"\nNOTE: only {len(sessions)} sessions loaded. DB6 spans 5 days x 2 sessions;"
        "\n      a cross-session result over this range understates drift by construction."
    )

## 2. The task, and the honest reduction

Reborn's decision is binary — assist flexion or do not. DB6 labels seven grasps
plus rest. The binary task is derived as **rest vs. any movement**, which is the
closest honest mapping onto the elbow orthosis; the multi-class task is reported
alongside so the reduction is visible rather than assumed
(`docs/research/phase-b-plan.md` §3).

Class balance matters for reading the numbers below: if rest is a quarter of
windows, a model that always predicts "movement" already scores 75% accuracy.
That is why balanced accuracy is the headline column.

In [ ]:
X_raw, feature_names = feature_matrix(window_set)
y_multi = window_set.labels
y_binary = window_set.binary_labels()

print(f"feature matrix: {X_raw.shape}  ({len(feature_names)} columns)")
print(f"columns: {feature_names}\n")

for name, y in (("multi-class", y_multi), ("binary (rest vs movement)", y_binary)):
    labels, counts = np.unique(y, return_counts=True)
    shares = "  ".join(f"{lab}:{n / len(y):.1%}" for lab, n in zip(labels, counts))
    print(f"{name:<28}{len(labels)} classes   {shares}")
    print(f"{'':<28}majority-class accuracy = {counts.max() / len(y):.1%}")

## 3. Models

LDA and calibrated logistic regression, both CPU-only. Deliberately simple: the
question this phase asks is what happens to *confidence* under drift, and a model
whose calibration can be reasoned about is worth more here than a point of
accuracy. A 1D CNN is warranted only if these turn out to be the limiting factor.

Two things the `fit_predict` closures below get right, and which are easy to get
wrong:

- **Standardisation is fitted on training rows only.** Scaling with statistics
  pooled over train and test leaks the per-session amplitude shift — which is
  precisely the drift under study — and flatters every cross-session number.
- **Confidence is the probability of the predicted class**, which is what
  `ConfidenceGate` consumes at runtime. LDA's posterior is used directly; nothing
  is rescaled to look better.

In [ ]:
def make_fit_predict(estimator_factory):
    """Wrap an sklearn estimator as the harness's (X_tr, y_tr, X_te) -> (pred, conf)."""

    def fit_predict(X_train, y_train, X_test):
        X_train, X_test = standardize(X_train, X_test)
        model = estimator_factory().fit(X_train, y_train)
        proba = model.predict_proba(X_test)
        predictions = model.classes_[np.argmax(proba, axis=1)]
        confidence = np.max(proba, axis=1)
        return predictions, confidence

    return fit_predict


MODELS = {
    "lda": make_fit_predict(LinearDiscriminantAnalysis),
    "logreg": make_fit_predict(lambda: LogisticRegression(max_iter=2000)),
}
print("models:", list(MODELS))

## 4. The four protocols

Each protocol yields several splits (one per session, per subject, and so on).
They are evaluated individually and summarised with **spread, not just a mean** —
under cross-session the variation between held-out sessions *is* the drift, and
averaging it away discards the result.

In [ ]:
protocols = {
    "within-session": within_session_splits(window_set),
    "cross-session": cross_session_splits(window_set, n_train_sessions=1),
    "cross-subject": cross_subject_splits(window_set),
    "random-shuffle": [random_window_split(window_set, seed=0)],
}

for name, splits in protocols.items():
    print(f"{name:<18}{len(splits):>3} splits")
    if not splits:
        print(f"{'':<18}    (none — needs more subjects or sessions than are loaded)")

In [ ]:
TASK = "binary"          # "binary" or "multi"
MODEL = "lda"            # key into MODELS

y = y_binary if TASK == "binary" else y_multi
fit_predict = MODELS[MODEL]

all_results = []
started = time.time()
for name, splits in protocols.items():
    if not splits:
        continue
    print(f"\n{name}  ({len(splits)} splits)")
    results = evaluate_splits(X_raw, y, splits, fit_predict, protocol=name, verbose=True)
    all_results.extend(results)
print(f"\n({time.time() - started:.0f} s)")

## 5. The baseline table

Read the **within-session → cross-session** gap first: that is the drift, and it
is what notebook 03 builds on.

Two ways this can come out flat, and they are not the same finding:

- **Sessions too close together.** DB6 spans 5 days, two sessions per day. If
  `SESSIONS` covers only day 1 and day 2, there is barely any elapsed time for
  drift to accumulate, and a small gap says nothing about the dataset. Check §6
  before concluding anything.
- **Genuinely little drift across the full span.** That is a real result, and it
  is the one the stop rule is about.

> **Stop rule** (`docs/research/checklist.md`): if cross-session is
> indistinguishable from within-session *across the full session range*, there is
> no drift to study here and the paper's framing needs revisiting before
> continuing to notebook 03. Widening `SESSIONS` is the first thing to try, not
> the conclusion.

In [ ]:
summary_rows = []
print(f"{'protocol':<18}{'splits':>7}{'bal.acc':>10}{'(spread)':>12}{'acc':>9}{'ECE':>8}{'unsafe':>9}")
print("-" * 74)
for name in protocols:
    subset = [r for r in all_results if r.protocol == name]
    if not subset:
        continue
    stats = summarize(subset)
    summary_rows.append({"protocol": name, "task": TASK, "model": MODEL, **stats})
    spread = f"{stats['balanced_accuracy_min']:.3f}-{stats['balanced_accuracy_max']:.3f}"
    print(
        f"{name:<18}{stats['n_splits']:>7.0f}{stats['balanced_accuracy_mean']:>10.3f}"
        f"{spread:>12}{stats['accuracy_mean']:>9.3f}"
        f"{stats['ece_mean']:>8.3f}{stats['unsafe_assist_rate_mean']:>9.3f}"
    )

within = next((r for r in summary_rows if r["protocol"] == "within-session"), None)
cross = next((r for r in summary_rows if r["protocol"] == "cross-session"), None)
shuffled = next((r for r in summary_rows if r["protocol"] == "random-shuffle"), None)

print()
if within and cross:
    drop = within["balanced_accuracy_mean"] - cross["balanced_accuracy_mean"]
    print(f"within-session -> cross-session: {drop:+.3f} balanced accuracy")
if shuffled and cross:
    inflation = shuffled["balanced_accuracy_mean"] - cross["balanced_accuracy_mean"]
    print(f"random-shuffle inflation over cross-session: {inflation:+.3f}")

## 6. Degradation against elapsed sessions

Cross-session splits carry `sessions_elapsed` — how far the test session sits
from the training one. If drift is real and cumulative, performance should fall
as this grows; if it is a fixed per-session offset, it should drop once and then
sit flat. Those are different physical stories and they call for different
personalization strategies in notebook 03, so the shape matters more than the
mean.

In [ ]:
by_elapsed = {}
for result in all_results:
    if result.protocol != "cross-session":
        continue
    by_elapsed.setdefault(result.meta.get("sessions_elapsed"), []).append(result)

if by_elapsed:
    print(f"{'elapsed':>8}{'splits':>8}{'bal.acc':>10}{'ECE':>8}{'unsafe':>9}")
    print("-" * 43)
    for elapsed in sorted(k for k in by_elapsed if k is not None):
        subset = by_elapsed[elapsed]
        stats = summarize(subset)
        print(
            f"{elapsed:>8}{len(subset):>8}{stats['balanced_accuracy_mean']:>10.3f}"
            f"{stats['ece_mean']:>8.3f}{stats['unsafe_assist_rate_mean']:>9.3f}"
        )
else:
    print("no cross-session splits — load more sessions")

## 7. Artifacts

Per-split rows and the protocol summary, both keyed by the preprocessing
fingerprint. Figures come from a separate script reading these CSVs, so no number
in the paper depends on a live kernel (`docs/research/phase-b-plan.md` §8).

In [ ]:
def write_csv(path, rows):
    if not rows:
        print(f"skipped {path.name}: nothing to write")
        return
    fields = sorted({key for row in rows for key in row})
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    print(f"wrote {path.relative_to(REPO)}  ({len(rows)} rows)")


stamp = f"{TASK}_{MODEL}_{config.fingerprint()}"
write_csv(RESULTS / f"nb02_splits_{stamp}.csv", [r.as_row() for r in all_results])
write_csv(RESULTS / f"nb02_summary_{stamp}.csv", summary_rows)

(RESULTS / f"nb02_run_{stamp}.json").write_text(
    json.dumps(
        {
            "config_fingerprint": config.fingerprint(),
            "dataset": "ninapro_db6",
            "subjects": SUBJECTS,
            "sessions": SESSIONS or "all",
            "montage_raw_columns": list(MONTAGE),
            "task": TASK,
            "model": MODEL,
            "features": list(FEATURE_NAMES),
            "n_windows": window_set.n_windows,
            "qc_rejection_rate": window_set.qc.rejection_rate,
        },
        indent=2,
    )
)
print(f"wrote experiments/results/nb02_run_{stamp}.json")

## 8. Reading & next steps

### Notes from this run

> _Written against the outputs above, not in advance. A number that changes no
> design decision does not need a note (`docs/experiments.md`)._

- **Within-session ceiling (§5)** — TBD
- **Cross-session drop (§5)** — TBD
- **Random-shuffle inflation (§5)** — TBD
- **Cross-subject worst case (§5)** — TBD
- **Shape of degradation vs. elapsed sessions (§6)** — TBD
- **Binary vs. multi-class (§2, rerun with `TASK`)** — TBD
- **LDA vs. logistic regression (rerun with `MODEL`)** — TBD

### Next

1. Rerun with `TASK = "multi"` and with `MODEL = "logreg"`; four runs total, and
   the artifacts are keyed so they do not overwrite each other.
2. If the cross-session drop is real, `03_drift_fewshot` asks the question this
   notebook does not: does confidence *fall with* accuracy, or stay high while
   accuracy falls? The second is the case that opens the gate on bad predictions.
3. If cross-session is indistinguishable from within-session, stop and revisit the
   framing before building on it (`docs/research/checklist.md` stop rule).